In [1]:
import pandas as pd
import lightgbm as lgb
import os
import sys

In [2]:
sys.path.append(os.path.abspath('src'))
from feature_engineering import create_features
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error, mean_absolute_error

In [3]:
print("Loading data for LightGBM...")
df = pd.read_csv('data/bike_sharing.csv')
df = create_features(df)

Loading data for LightGBM...


In [4]:
features = [
    'hour', 'temp', 'is_holiday', 'day_of_week', 'month', 'quarter',
    'is_weekend', 'lag_7', 'lag_14', 'lag_30',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_30'
]
target = 'demand'

In [5]:
X = df[features]
y = df[target]

In [6]:
print("Evaluating using TimeSeriesSplit...")
tscv = TimeSeriesSplit(n_splits=5)

Evaluating using TimeSeriesSplit...


In [7]:
mapes = []
rmses = []
maes = []

In [8]:
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=7, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mapes.append(mean_absolute_percentage_error(y_test, preds))
    rmses.append(root_mean_squared_error(y_test, preds))
    maes.append(mean_absolute_error(y_test, preds))

In [9]:
print(f"LightGBM -> MAPE: {sum(mapes)/len(mapes):.4f}, RMSE: {sum(rmses)/len(rmses):.4f}, MAE: {sum(maes)/len(maes):.4f}")

LightGBM -> MAPE: 0.0853, RMSE: 12.5996, MAE: 10.1147
